# Comparing Text-to-Speech Audio Quality

by Tobias Erbacher

Let's first start by obtaining information about the environment in which this notebook is run.

In [1]:
import os
RUNNING_ON_LOCAL_MACHINE = True if 'COLAB_GPU' not in os.environ else False

import torch
GPU_AVAILABLE = torch.cuda.is_available()

In [2]:
from IPython.display import Audio

if RUNNING_ON_LOCAL_MACHINE:
  !pip install pyttsx3
  import pyttsx3
  import threading

if GPU_AVAILABLE:
  !pip install inflect unidecode
  import unidecode
  import inflect

!pip install gTTS
from gtts import gTTS
import io

We define a common sentence that we have each model say:

In [3]:
TEXT = "Low dietary calcium is associated with a higher risk of calcium oxalate stones."

Next, set up the models. In particular, we will test automatically generated voices:

- PyTTSx3 (local machine only)
- Tacotron 2 + WaveGlow
- gTTS

Moreover, we will also investigate custom voices:

- Tacotron 2 + WaveNet/Vocoder
- FastSpeech 2
- VITS (Variational Inference Text-to-Speech)

---

### PyTTSx3

In [ ]:
def pyttsx3(text, voice=132):
  SHOW_AVAILABLE_VOICES = False
  VOICE = voice
  TALKING_SPEED = 150
  VOLUME = 1

  def pyttsx3_speak(engine, text, event):
    VOICES = engine.getProperty('voices')
    if SHOW_AVAILABLE_VOICES:
      for idx, voice in enumerate(VOICES):
        print("Voice: " + str(idx) + ", ID: " + str(voice.id) + ", Name: " + str(voice.name) + ", Language: " + str(voice.languages))
    engine.setProperty('voice', VOICES[VOICE].id)
    engine.setProperty('rate', TALKING_SPEED)
    engine.setProperty('volume', VOLUME)
    try:
      if engine._inLoop:
        engine.endLoop()
      engine.say(text)
      event.set()
      engine.runAndWait()
    except:
      engine.stop()
      engine.endLoop()
      print("Response interrupted.")

  def pyttsx3_initEngine(callback, text, event):
    global PYTTSX3_ENGINE
    PYTTSX3_ENGINE = pyttsx3.init()
    print("Playing system response...")
    callback(PYTTSX3_ENGINE, text, event)

  def pyttsx3_speakThread(text):
    thread_finished_event = threading.Event()
    thread = threading.Thread(target=pyttsx3_initEngine, args=(pyttsx3_speak, text, thread_finished_event))
    thread.start()
    thread_finished_event.wait()

  if RUNNING_ON_LOCAL_MACHINE:
    print("Waiting for system response...")
    pyttsx3_speakThread(text)
    print("System response generated.")
  else:
    print("Error: Cannot run this model outside of a local machine.")


Let us test the model:

In [ ]:
pyttsx3(TEXT)

---

### Tacotron2 + WaveGlow

In [6]:
def tacotron2_waveglow(text):
  WAVEGLOW_SAMPLING_RATE = 22050

  if GPU_AVAILABLE:
    TACOTRON2 = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_tacotron2', model_math='fp32').to('cuda')

    WAVEGLOW = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_waveglow', model_math='fp32')
    WAVEGLOW = WAVEGLOW.remove_weightnorm(WAVEGLOW).to('cuda')

    TTS_UTILS = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_tts_utils')
    WAVEGLOW_SEQUENCES, WAVEGLOW_LENGTHS = TTS_UTILS.prepare_input_sequence([text])

    with torch.no_grad():
      TACOTRON2_MEL, _, _ = TACOTRON2.infer(WAVEGLOW_SEQUENCES, WAVEGLOW_LENGTHS)
      TACOTRON2_VOICE = WAVEGLOW.infer(TACOTRON2_MEL)[0].data.cpu().numpy()

      return Audio(TACOTRON2_VOICE, rate=WAVEGLOW_SAMPLING_RATE)
  else:
    print("Error: Cannot run this model without GPU.")

Let us test the model:

In [7]:
tacotron2_waveglow(TEXT)

Using cache found in /root/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub
Using cache found in /root/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub
Using cache found in /root/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub


---

### gTTS

In [18]:
def gtts(text):
  tts = gTTS(text, lang="en", tld='us')
  buffer = io.BytesIO()
  tts.write_to_fp(buffer)
  buffer.seek(0)
  return Audio(buffer.read(), autoplay=True)

Let us test the model:

In [19]:
gtts(TEXT)

---

### Tacotron 2 + WaveNet/Vocoder

---

### FastSpeech 2

---

### VITS (Variational Inference Text-to-Speech)